# Week 2 MongoDB CRUD and Dash Project

This notebook extends the original sales dashboard by loading the CSV data into MongoDB, performing CRUD operations, documenting the changes, and visualizing the updated totals with Dash.

Before running the MongoDB cells, replace `YOUR_MONGODB_CONNECTION_STRING` with your own MongoDB connection string.

In [ ]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html
from pymongo import MongoClient
from IPython.display import display

## 1. Load the Sales Dataset

The CSV file contains the required fields: `date`, `product_id`, `sales_amount`, and `store_location`.

In [ ]:
sales_data = pd.read_csv('sales_data.csv')
sales_data['date'] = pd.to_datetime(sales_data['date'])

print('Shape:', sales_data.shape)
print('Columns:', sales_data.columns.tolist())
display(sales_data.head())

## 2. Insert the CSV Data into MongoDB

The code below connects to MongoDB, creates the `sales_db` database and `sales` collection, clears any previous records, and inserts the CSV rows so the notebook can be rerun cleanly.

In [ ]:
MONGO_URI = 'YOUR_MONGODB_CONNECTION_STRING'

client = MongoClient(MONGO_URI)
db = client['sales_db']
sales_collection = db['sales']

sales_collection.delete_many({})

records = sales_data.to_dict('records')
insert_result = sales_collection.insert_many(records)

crud_log = {
    'initial_csv_records': len(records),
    'create': [],
    'read': {},
    'update': [],
    'delete': {}
}

print(f"Inserted {len(insert_result.inserted_ids)} records into sales_db.sales")

## 3. Perform CRUD Operations

The following cells demonstrate create, read, update, and delete operations on the MongoDB collection.

In [ ]:
# Create: add two new sales records
new_sales_records = [
    {
        'date': pd.Timestamp('2023-09-17').to_pydatetime(),
        'product_id': 'P007',
        'sales_amount': 610,
        'store_location': 'Boston'
    },
    {
        'date': pd.Timestamp('2023-09-17').to_pydatetime(),
        'product_id': 'P002',
        'sales_amount': 390,
        'store_location': 'New York'
    }
]

create_result = sales_collection.insert_many(new_sales_records)
crud_log['create'] = new_sales_records

print(f"Inserted {len(create_result.inserted_ids)} additional records.")
display(pd.DataFrame(new_sales_records))

In [ ]:
# Read: query records by store location, date, and product
location_query = list(
    sales_collection.find(
        {'store_location': 'New York'},
        {'_id': 0}
    )
)

date_query = list(
    sales_collection.find(
        {'date': pd.Timestamp('2023-09-10').to_pydatetime()},
        {'_id': 0}
    )
)

product_query = list(
    sales_collection.find(
        {'product_id': 'P002'},
        {'_id': 0}
    )
)

crud_log['read'] = {
    'new_york_records': len(location_query),
    'records_on_2023_09_10': len(date_query),
    'product_p002_records': len(product_query)
}

print('Records from New York:')
display(pd.DataFrame(location_query))

print('Records on 2023-09-10:')
display(pd.DataFrame(date_query))

print('Records for product P002:')
display(pd.DataFrame(product_query))

In [ ]:
# Update: increase sales for product P001 by 10%
target_product = 'P001'
records_to_update = list(sales_collection.find({'product_id': target_product}))

for record in records_to_update:
    old_amount = record['sales_amount']
    new_amount = round(old_amount * 1.10, 2)
    sales_collection.update_one(
        {'_id': record['_id']},
        {'$set': {'sales_amount': new_amount}}
    )
    crud_log['update'].append({
        'product_id': target_product,
        'date': record['date'],
        'store_location': record['store_location'],
        'old_sales_amount': old_amount,
        'new_sales_amount': new_amount
    })

updated_preview = pd.DataFrame(crud_log['update'])
print(f"Updated {len(updated_preview)} {target_product} records by increasing sales_amount by 10%.")
display(updated_preview.head())

In [ ]:
# Delete: remove outdated records before 2023-09-03
cutoff_date = pd.Timestamp('2023-09-03').to_pydatetime()
records_to_delete = list(
    sales_collection.find(
        {'date': {'$lt': cutoff_date}},
        {'_id': 0}
    )
)

delete_result = sales_collection.delete_many({'date': {'$lt': cutoff_date}})

crud_log['delete'] = {
    'cutoff_date': '2023-09-03',
    'deleted_count': delete_result.deleted_count,
    'deleted_records_preview': records_to_delete[:5]
}

print(f"Deleted {delete_result.deleted_count} outdated records.")
display(pd.DataFrame(records_to_delete))

## 4. Document the CRUD Operations

CRUD operations are important because they represent the core actions needed to manage data in a database:

- **Create** adds new business records.
- **Read** retrieves targeted information for analysis.
- **Update** keeps existing data accurate and current.
- **Delete** removes outdated or unnecessary records.

In this notebook, the sales collection is first loaded from the CSV file, then expanded with new records, queried in different ways, updated for one product, and cleaned by removing older records.

In [ ]:
documentation_summary = pd.DataFrame([
    {
        'operation': 'Initial Load',
        'details': f"Inserted {crud_log['initial_csv_records']} records from the CSV file."
    },
    {
        'operation': 'Create',
        'details': f"Added {len(crud_log['create'])} new sales records."
    },
    {
        'operation': 'Read',
        'details': (
            f"Queried {crud_log['read']['new_york_records']} New York records, "
            f"{crud_log['read']['records_on_2023_09_10']} records on 2023-09-10, "
            f"and {crud_log['read']['product_p002_records']} P002 records."
        )
    },
    {
        'operation': 'Update',
        'details': f"Increased sales_amount by 10% for {len(crud_log['update'])} P001 records."
    },
    {
        'operation': 'Delete',
        'details': f"Deleted {crud_log['delete']['deleted_count']} records dated before {crud_log['delete']['cutoff_date']}."
    },
    {
        'operation': 'Final Record Count',
        'details': f"The collection now contains {sales_collection.count_documents({})} records after CRUD operations."
    }
])

display(documentation_summary)

## 5. Visualize the Updated MongoDB Data with Dash

The bar chart below is created from the updated MongoDB collection after all CRUD operations have been applied.

In [ ]:
updated_sales = pd.DataFrame(list(sales_collection.find({}, {'_id': 0})))
updated_sales['date'] = pd.to_datetime(updated_sales['date'])

sales_by_location = (
    updated_sales.groupby('store_location', as_index=False)['sales_amount']
    .sum()
    .sort_values('sales_amount', ascending=False)
)

display(sales_by_location)

app = Dash(__name__)

fig = px.bar(
    sales_by_location,
    x='store_location',
    y='sales_amount',
    color='store_location',
    title='Total Sales by Store Location After MongoDB CRUD Operations',
    labels={
        'store_location': 'Store Location',
        'sales_amount': 'Total Sales Amount'
    }
)
fig.update_layout(template='plotly_white', showlegend=False)

app.layout = html.Div(
    children=[
        html.H1('Updated Sales Dashboard', style={'textAlign': 'center'}),
        html.P(
            'MongoDB-backed sales totals after create, read, update, and delete operations.',
            style={'textAlign': 'center'}
        ),
        dcc.Graph(figure=fig)
    ],
    style={'fontFamily': 'Arial, sans-serif', 'padding': '20px'}
)

In [ ]:
# Run the Dash application after setting your MongoDB connection string.
app.run(debug=True, port=8051)